# Phase 1 — Data Ingestion

This notebook exercises `src/ingest.py` to pull every data source we'll need for the FanTeasy Stats pipeline. Each fetch is cached to `data/raw/` so re-running is fast.

**What we pull:**

| Source | What it gives us | Used in phase |
|---|---|---|
| nflverse play-by-play | Every play with air yards, pass location, etc. | 2 (features) + 5 (heatmaps) |
| nflverse weekly stats | Per-week fantasy totals per player | 2 + 6 |
| nflverse snap counts | Offensive/ST snap participation | 2 + 3 (roles) |
| nflverse NGS | aDOT, separation, time-to-throw | 2 + 4 (radar) |
| nflverse schedule | Weather, roof, matchup context | 6 (projections) |
| nflverse rosters | Player-team-season assignments | 2 |
| nflverse ID crosswalk | gsis_id ↔ sleeper_id joiner | 7 (export) |
| Sleeper players | Injury status, depth chart, sleeper_id | 7 |
| Sleeper league | Scoring settings, roster positions | 2 (custom scoring) |
| Sleeper projections | Baseline to beat with our model | 6 |


## Setup

In [1]:
# Add project root to path so we can import from src/
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Verbose logging so we can see cache hits/misses
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

In [2]:
from src.ingest import (
    get_pbp, get_weekly_stats, get_snap_counts,
    get_ngs_data, get_schedule, get_seasonal_rosters,
    get_id_crosswalk,
    get_sleeper_league, get_sleeper_players, get_sleeper_projections,
    DEFAULT_LEAGUE_ID,
)

## Choose which season(s) to pull

For a single-season model, 1 year is enough. For anything predictive it's worth pulling at least 2 so the model sees week-to-week variance across years.

> **Note**: `nflreadpy` season data becomes available a few days after each week's games. If the current season is very early or hasn't started yet, only pull past seasons.

In [3]:
# Adjust as the season progresses. 2024 = full historical, 2025 = current.
SEASONS = [2024, 2025]

## 1. Sleeper league config

Small fetch, but everything else depends on knowing the league's scoring rules and roster structure.

In [4]:
league = get_sleeper_league(DEFAULT_LEAGUE_ID)
print(f"League: {league.get('name')}  ({league.get('season')} season)")
print(f"Teams: {league.get('total_rosters')}")
print(f"Scoring format: {league.get('scoring_settings', {}).get('rec', 0)}pt PPR")
print(f"Roster slots: {league.get('roster_positions')}")

# Sleeper mints a new league_id per season for dynasty leagues. If the season
# above isn't the one you're modeling, this is last year's league — grab the
# current id from Sleeper and update DEFAULT_LEAGUE_ID in src/ingest.py.
print(f"\nprevious_league_id: {league.get('previous_league_id')}")

[cache hit]  sleeper_league_1389706592789733376.json


League: Fanteasy Football  (2026 season)
Teams: 14
Scoring format: 0.5pt PPR
Roster slots: ['QB', 'RB', 'RB', 'WR', 'WR', 'TE', 'FLEX', 'K', 'DEF', 'BN', 'BN', 'BN', 'BN', 'BN']

previous_league_id: 1250182471429931008


In [5]:
# Inspect the full scoring settings — this drives custom fantasy point
# calculations in Phase 2. Any field here has a stat with the same name
# in the weekly/pbp data.
scoring = league.get('scoring_settings', {})
scoring_df = pd.DataFrame([
    {'setting': k, 'value': v}
    for k, v in scoring.items() if isinstance(v, (int, float)) and v != 0
]).sort_values('setting')
print(f"{len(scoring_df)} non-zero scoring settings:")
scoring_df

52 non-zero scoring settings:


,setting,value
38,blk_kick,2.000000
30,def_3_and_out,0.250000
11,def_4_and_stop,0.500000
49,def_st_ff,1.000000
24,def_st_fum_rec,1.000000
34,def_st_td,6.000000
35,def_td,6.000000
19,ff,1.000000
14,fgm,3.000000
6,fgm_yds_over_30,0.100000


## 2. Sleeper player DB

The full NFL player registry from Sleeper's side. Provides the `sleeper_id` we'll key everything by in the final JSON export.

In [6]:
sleeper_players = get_sleeper_players()
print(f"Total Sleeper player entries: {len(sleeper_players):,}")
sleeper_players.head()

[cache hit]  sleeper_players.json


Total Sleeper player entries: 12,217


,sleeper_id,first_name,last_name,full_name,position,team,age,years_exp,status,injury_status,injury_body_part,injury_notes,injury_start_date,depth_chart_position,depth_chart_order,fantasy_positions,college,height,weight
0,6462,Ellis,Richardson,Ellis Richardson,TE,NaN,26.0,3.0,Active,NaN,NaN,NaN,NaN,NaN,NaN,[TE],Georgia Southern,75,245
1,11255,Nick,Amoah,Nick Amoah,OL,NaN,NaN,3.0,Active,NaN,NaN,NaN,NaN,NaN,NaN,[OL],UC Davis,74,306
2,8842,Malkelm,Morrison,Malkelm Morrison,CB,NaN,NaN,1.0,Injured Reserve,NaN,NaN,NaN,NaN,NaN,NaN,[DB],Army,70,186
3,13940,Bruno,Fina,Bruno Fina,OL,BUF,NaN,0.0,Active,NaN,NaN,NaN,NaN,OL,NaN,[OL],Duke,77,305
4,7926,Carl,Tucker,Carl Tucker,TE,NaN,24.0,1.0,Active,NaN,NaN,NaN,NaN,NaN,NaN,[TE],Alabama,74,250


In [7]:
# Quick sanity checks — how many active fantasy-relevant players?
active_fantasy = sleeper_players[
    (sleeper_players['position'].isin(['QB', 'RB', 'WR', 'TE', 'K', 'DEF']))
    & (sleeper_players['status'] == 'Active')
]
print(f"Active fantasy-relevant players: {len(active_fantasy):,}")
print(active_fantasy['position'].value_counts().to_string())

Active fantasy-relevant players: 2,943
position
WR    1253
RB     628
TE     575
QB     339
K      148


## 3. Player ID crosswalk

The **most important** table for later phases. nflverse's `gsis_id` and Sleeper's `sleeper_id` don't match — this table bridges them plus a handful of other systems (ESPN, Yahoo, PFR, PFF).

In [8]:
crosswalk = get_id_crosswalk()
print(f"Total crosswalk rows: {len(crosswalk):,}")
print(f"Columns available: {list(crosswalk.columns)}")
crosswalk.head()

[cache hit]  id_crosswalk.parquet


Total crosswalk rows: 12,470
Columns available: ['mfl_id', 'sportradar_id', 'fantasypros_id', 'gsis_id', 'pff_id', 'sleeper_id', 'nfl_id', 'espn_id', 'yahoo_id', 'fleaflicker_id', 'cbs_id', 'pfr_id', 'cfbref_id', 'rotowire_id', 'rotoworld_id', 'ktc_id', 'stats_id', 'stats_global_id', 'fantasy_data_id', 'swish_id', 'name', 'merge_name', 'position', 'team', 'birthdate', 'age', 'draft_year', 'draft_round', 'draft_pick', 'draft_ovr', 'twitter_username', 'height', 'weight', 'college', 'db_season']


,mfl_id,sportradar_id,fantasypros_id,gsis_id,pff_id,sleeper_id,nfl_id,espn_id,yahoo_id,fleaflicker_id,cbs_id,pfr_id,cfbref_id,rotowire_id,rotoworld_id,...,name,merge_name,position,team,birthdate,age,draft_year,draft_round,draft_pick,draft_ovr,twitter_username,height,weight,college,db_season
0,17462,b1ded115-092a-4199-9a55-cab9f4b5bb18,28013.0,00-0041562,NaN,13269,62623.0,4837248,<NA>,NaN,28916321.0,MendFe00,fernando-mendoza-1,19281.0,NaN,...,Fernando Mendoza,fernando mendoza,QB,LVR,2003-10-01,22.9,2026.0,1.0,1.0,1.0,NaN,77.0,225.0,Indiana,2026
1,17463,2e62e603-363a-425f-8c57-0a1600476c85,25368.0,00-0041568,NaN,13275,62635.0,4685522,<NA>,NaN,28878532.0,SimpTy00,ty-simpson-1,19275.0,NaN,...,Ty Simpson,ty simpson,QB,LAR,2002-12-21,23.6,2026.0,1.0,13.0,13.0,NaN,74.0,203.0,Alabama,2026
2,17464,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,...,Trinidad Chambliss,trinidad chambliss,XX,FA,NaN,NaN,2026.0,NaN,NaN,NaN,NaN,NaN,NaN,Mississippi,2026
3,17465,9fc8bed4-d9db-46a2-a75f-737881d23aa7,22998.0,00-0040906,NaN,13404,62871.0,4567747,<NA>,NaN,26701706.0,NussGa00,garrett-nussmeier-1,19283.0,NaN,...,Garrett Nussmeier,garrett nussmeier,QB,KCC,2002-02-06,24.5,2026.0,7.0,33.0,249.0,NaN,73.0,205.0,LSU,2026
4,17466,a67c28ce-9e03-46f1-9fb6-b5ef70eb20db,22953.0,00-0041561,NaN,13272,62687.0,4430841,<NA>,NaN,3163129.0,BeckCa01,carson-beck-1,19287.0,NaN,...,Carson Beck,carson beck,QB,ARI,2001-11-19,24.7,2026.0,3.0,1.0,65.0,NaN,77.0,233.0,Miami (Fla.),2026


In [9]:
# How complete is the sleeper_id column? Some historical players
# without Sleeper accounts will have NaN — that's fine, they can't be
# in our export anyway.
with_sleeper = crosswalk['sleeper_id'].notna().sum()
print(f"Rows with sleeper_id: {with_sleeper:,} / {len(crosswalk):,}")
print(f"Rows with gsis_id:    {crosswalk['gsis_id'].notna().sum():,}")

# Save a slim version we'll use for join lookups later
from src.ingest import DATA_PROCESSED
slim = crosswalk[['gsis_id', 'sleeper_id', 'name', 'position', 'team']].dropna(subset=['sleeper_id'])
out_path = DATA_PROCESSED / 'id_crosswalk_slim.csv'
slim.to_csv(out_path, index=False)
print(f"\nSlim crosswalk saved: {out_path}  ({len(slim):,} rows)")

Rows with sleeper_id: 6,362 / 12,470
Rows with gsis_id:    7,988

Slim crosswalk saved: C:\Users\rohbh\Claude Projects\fanteasy-notebook\fanteasy-notebook\data\processed\id_crosswalk_slim.csv  (6,362 rows)


## 4. Weekly stats (aggregated by player-week)

Easier to work with than raw pbp for most feature computations. Use this for anything that doesn't need play-level detail.

In [10]:
weekly = get_weekly_stats(SEASONS)
print(f"Weekly stat rows: {len(weekly):,}")
print(f"Columns: {list(weekly.columns)}")
weekly.head()

[cache hit]  weekly_2024_2025.parquet


Weekly stat rows: 38,402
Columns: ['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'recei

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,...,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_NYJ_SF,NYJ,SF,13,21,167,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,8.58,8.58
1,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_ARI_BUF,ARI,BUF,0,0,0,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.00,0.00
2,00-0025565,N.Folk,Nick Folk,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_TEN_CHI,TEN,CHI,0,0,0,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.00,0.00
3,00-0026190,C.Campbell,Calais Campbell,DE,DL,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_JAX_MIA,MIA,JAX,0,0,0,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0.00,0.00
4,00-0026498,M.Stafford,Matthew Stafford,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_LA_DET,LA,DET,34,49,317,...,0,0,NaN,0,0,0,0,0,0,0,0,0,0,14.68,14.68


In [11]:
# Spot-check: highest fantasy scorers in the most recent complete week
latest_season = max(SEASONS)
latest_wk = weekly[weekly['season'] == latest_season]['week'].max()
print(f"Latest week in data: {latest_season} Wk {latest_wk}")

# nflverse renamed this column between stat releases ('recent_team' -> 'team').
# Resolve whichever exists so this cell survives the next schema bump.
TEAM_COL = next(c for c in ['team', 'recent_team'] if c in weekly.columns)
print(f"Using team column: {TEAM_COL}")

top10 = (weekly[(weekly['season'] == latest_season) & (weekly['week'] == latest_wk)]
         .sort_values('fantasy_points_ppr', ascending=False)
         .head(10)[['player_display_name', 'position', TEAM_COL,
                    'fantasy_points_ppr', 'fantasy_points']])
top10

Latest week in data: 2025 Wk 22
Using team column: team


,player_display_name,position,team,fantasy_points_ppr,fantasy_points
38371,Kenneth Walker III,RB,SEA,18.10,16.10
38342,Mack Hollins,WR,NE,17.80,13.80
38391,Drake Maye,QB,NE,17.50,17.50
38358,Rhamondre Stevenson,RB,NE,17.30,12.30
38389,AJ Barner,TE,SEA,15.40,11.40
38349,Sam Darnold,QB,SEA,12.58,12.58
38343,Cooper Kupp,WR,SEA,12.10,6.10
38377,DeMario Douglas,WR,NE,9.50,4.50
38400,TreVeyon Henderson,RB,NE,7.50,4.50
38337,Stefon Diggs,WR,NE,6.70,3.70


## 5. Play-by-play

The big one — ~50k rows per season. Only fetch this if you need play-level fields (air_yards, pass_location, run_gap, etc.) which you'll need for radar metrics and heatmaps.

In [12]:
pbp = get_pbp(SEASONS)
print(f"Play-by-play rows: {len(pbp):,}")
# Skip printing all columns — there are 300+ — just show the ones
# we'll actually use in Phase 2
phase2_cols = [c for c in pbp.columns if c in [
    'passer_player_id', 'receiver_player_id', 'rusher_player_id',
    'passing_yards', 'receiving_yards', 'rushing_yards',
    'pass_touchdown', 'rush_touchdown',
    'air_yards', 'yards_after_catch',
    'pass_location', 'run_location', 'run_gap',
    'yardline_100', 'complete_pass', 'interception', 'sack',
    'week', 'season', 'posteam', 'defteam',
]]
pbp[phase2_cols].head()

[cache hit]  pbp_2024_2025.parquet


Play-by-play rows: 98,263


,week,posteam,defteam,yardline_100,pass_location,air_yards,yards_after_catch,run_location,run_gap,interception,sack,pass_touchdown,rush_touchdown,complete_pass,passer_player_id,passing_yards,receiver_player_id,receiving_yards,rusher_player_id,rushing_yards,season
0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
1,1,ARI,BUF,35.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,2024
2,1,ARI,BUF,70.0,NaN,NaN,NaN,middle,NaN,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,00-0033553,3.0,2024
3,1,ARI,BUF,67.0,left,-3.0,25.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,00-0035228,22.0,00-0033553,22.0,NaN,NaN,2024
4,1,ARI,BUF,45.0,middle,2.0,7.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,00-0035228,9.0,00-0033553,9.0,NaN,NaN,2024


## 6. Snap counts

Critical for role classification: is this RB a 3-down back (65%+ snap share) or a committee back?

In [13]:
snaps = get_snap_counts(SEASONS)
print(f"Snap-count rows: {len(snaps):,}")
snaps.head()

[cache hit]  snaps_2024_2025.parquet


Snap-count rows: 53,227


,game_id,pfr_game_id,season,game_type,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
0,2024_01_ARI_BUF,202409080buf,2024,REG,1,Spencer Brown,BrowSp00,T,BUF,ARI,62.0,1.0,0.0,0.0,6.0,0.22
1,2024_01_ARI_BUF,202409080buf,2024,REG,1,O'Cyrus Torrence,TorrOC00,G,BUF,ARI,62.0,1.0,0.0,0.0,6.0,0.22
2,2024_01_ARI_BUF,202409080buf,2024,REG,1,Dion Dawkins,DawkDi00,T,BUF,ARI,62.0,1.0,0.0,0.0,6.0,0.22
3,2024_01_ARI_BUF,202409080buf,2024,REG,1,David Edwards,EdwaDa01,G,BUF,ARI,62.0,1.0,0.0,0.0,6.0,0.22
4,2024_01_ARI_BUF,202409080buf,2024,REG,1,Josh Allen,AlleJo02,QB,BUF,ARI,62.0,1.0,0.0,0.0,0.0,0.00


## 7. Next Gen Stats

aDOT, separation, time-to-throw. Only available for players with tracking-data participation, so smaller universe than pbp.

In [14]:
ngs_receiving = get_ngs_data('receiving', SEASONS)
ngs_passing = get_ngs_data('passing', SEASONS)
ngs_rushing = get_ngs_data('rushing', SEASONS)

print(f"NGS receiving rows: {len(ngs_receiving):,}")
print(f"NGS passing rows:   {len(ngs_passing):,}")
print(f"NGS rushing rows:   {len(ngs_rushing):,}")

[cache hit]  ngs_receiving_2024_2025.parquet
[cache hit]  ngs_passing_2024_2025.parquet
[cache hit]  ngs_rushing_2024_2025.parquet


NGS receiving rows: 2,837
NGS passing rows:   1,219
NGS rushing rows:   1,249


## 8. Schedule (weather + matchup context)

Small table but useful for the projection model — dome vs outdoor, temperature, wind.

In [15]:
schedule = get_schedule(SEASONS)
print(f"Schedule rows: {len(schedule):,}")
schedule[['game_id', 'season', 'week', 'gameday', 'home_team', 'away_team',
          'roof', 'surface', 'temp', 'wind']].head(10)

[cache hit]  schedule_2024_2025.parquet


Schedule rows: 570


,game_id,season,week,gameday,home_team,away_team,roof,surface,temp,wind
0,2024_01_BAL_KC,2024,1,2024-09-05,KC,BAL,outdoors,grass,67.0,8.0
1,2024_01_GB_PHI,2024,1,2024-09-06,PHI,GB,outdoors,,NaN,NaN
2,2024_01_PIT_ATL,2024,1,2024-09-08,ATL,PIT,closed,fieldturf,NaN,NaN
3,2024_01_ARI_BUF,2024,1,2024-09-08,BUF,ARI,outdoors,a_turf,61.0,20.0
4,2024_01_TEN_CHI,2024,1,2024-09-08,CHI,TEN,outdoors,grass,67.0,8.0
5,2024_01_NE_CIN,2024,1,2024-09-08,CIN,NE,outdoors,fieldturf,66.0,5.0
6,2024_01_HOU_IND,2024,1,2024-09-08,IND,HOU,closed,fieldturf,NaN,NaN
7,2024_01_JAX_MIA,2024,1,2024-09-08,MIA,JAX,outdoors,grass,91.0,10.0
8,2024_01_CAR_NO,2024,1,2024-09-08,NO,CAR,dome,sportturf,NaN,NaN
9,2024_01_MIN_NYG,2024,1,2024-09-08,NYG,MIN,outdoors,fieldturf,64.0,10.0


## 9. Sleeper projections

The baseline we're trying to beat. Fetches per-week — pull the current week for reference and any past week you want to backtest against.

In [16]:
# Pick a week — adjust to whatever's current for your project
PROJ_SEASON = 2025
PROJ_WEEK = 1

proj = get_sleeper_projections(PROJ_SEASON, PROJ_WEEK)
print(f"Projection rows for {PROJ_SEASON} Wk {PROJ_WEEK}: {len(proj):,}")
print(f"Columns available: {list(proj.columns)[:20]}...")
proj.head()

[cache hit]  sleeper_proj_2025_wk1.json


Projection rows for 2025 Wk 1: 9,411
Columns available: ['sleeper_id', 'adp_dd_ppr', 'fga', 'fgm', 'fgm_20_29', 'fgm_30_39', 'fgm_40_49', 'fgm_50p', 'fgm_yds', 'fgmiss_30_39', 'fgmiss_40_49', 'fgmiss_50p', 'gp', 'pos_adp_dd_ppr', 'pts_half_ppr', 'pts_ppr', 'pts_std', 'xpa', 'xpm', 'xpmiss']...


,sleeper_id,adp_dd_ppr,fga,fgm,fgm_20_29,fgm_30_39,fgm_40_49,fgm_50p,fgm_yds,fgmiss_30_39,fgmiss_40_49,fgmiss_50p,gp,pos_adp_dd_ppr,pts_half_ppr,...,sack,safe,tkl_loss,yds_allow,yds_allow_350_399,def_kr_td,fgm_0_19,idp_fum_ret_yd,pts_allow_14_20,yds_allow_300_349,idp_safe,pr_td,st_td,fgmiss_20_29,yds_allow_400_449
0,6462,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11255,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8842,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,13940,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7926,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Sanity: end-to-end join test

Pick one player from Sleeper's DB and confirm we can join them to their nflverse stats via the crosswalk. If this works for one player, it works for all of them.

In [17]:
# Look up a well-known player (e.g. Josh Allen) by Sleeper ID
sample_sleeper_id = sleeper_players[
    sleeper_players['full_name'].str.contains('Josh Allen', na=False)
    & (sleeper_players['position'] == 'QB')
]['sleeper_id'].iloc[0]
print(f"Josh Allen's sleeper_id: {sample_sleeper_id!r}")

# Both sides are strings because get_id_crosswalk() normalizes ID columns.
# If this assert trips, that normalization is the first thing to check —
# a float 4984.0 will never match the string '4984'.
match = crosswalk[crosswalk['sleeper_id'] == sample_sleeper_id]
assert not match.empty, "Crosswalk miss — check sleeper_id dtype on both sides"
gsis = match['gsis_id'].iloc[0]
print(f"Josh Allen's gsis_id:    {gsis!r}")

# Pull all their weekly rows from nflverse
wanted = ['season', 'week', 'player_display_name', TEAM_COL,
          'completions', 'attempts', 'passing_yards', 'passing_tds',
          'interceptions', 'carries', 'rushing_yards', 'rushing_tds',
          'fantasy_points_ppr']
player_weekly = (weekly[weekly['player_id'] == gsis]
                 [[c for c in wanted if c in weekly.columns]]
                 .sort_values(['season', 'week']))
print(f"\nRows found in weekly stats: {len(player_weekly)}")
player_weekly.tail(10)

Josh Allen's sleeper_id: '4984'
Josh Allen's gsis_id:    '00-0034857'

Rows found in weekly stats: 37


,season,week,player_display_name,team,completions,attempts,passing_yards,passing_tds,carries,rushing_yards,rushing_tds,fantasy_points_ppr
28436,2025,10,Josh Allen,BUF,28,40,306,2,4,31,0,19.34
29398,2025,11,Josh Allen,BUF,19,30,317,3,6,40,3,42.68
30419,2025,12,Josh Allen,BUF,24,34,253,0,5,20,0,8.12
31416,2025,13,Josh Allen,BUF,15,23,123,1,8,38,1,16.72
32484,2025,14,Josh Allen,BUF,22,28,251,3,9,78,1,37.84
33460,2025,15,Josh Allen,BUF,19,28,193,3,11,48,0,24.52
34517,2025,16,Josh Allen,BUF,12,19,130,0,7,17,0,6.90
35608,2025,17,Josh Allen,BUF,23,35,262,0,7,27,2,23.18
37599,2025,19,Josh Allen,BUF,28,35,273,1,11,33,2,30.22
37977,2025,20,Josh Allen,BUF,25,39,283,3,12,66,0,21.92


## What's next

Everything above is now cached in `data/raw/` as parquet/json. Re-running any
`get_*()` function hits the cache and returns instantly.

**Phase 1 is complete and verified:**
- ID crosswalk joins nflverse ↔ Sleeper correctly (cell above)
- League config, weekly stats, pbp, snaps, NGS, schedule all pulling clean

**Phase 2 begins in `02_custom_scoring.ipynb`** — computing fantasy points the
way this league actually scores them, validated against Sleeper's own numbers.
That notebook produces the target variable everything downstream depends on.

Keep this notebook to ingestion only. Exploration and debugging belong in their
own notebook so this one stays re-runnable top to bottom.
